In [4]:
import os
from langchain_ollama import OllamaEmbeddings
from langchain.document_loaders import CSVLoader
import numpy as np
import faiss
import ollama
import tempfile
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")

In [5]:
data = pd.read_csv('vehicles-schema.csv')

In [6]:
def generate_suggestions(row):
    prompt = f'''genera un resumen corto y preciso de los siguientes datos de un vehiculo:
    {row.to_dict()}, guarda el dato en un solo parrafo iniciando con "Resumen: " donde hable de todas las columnas y sin usar comillas ni puntos y aparte, solo un parrafo.'''
    response = ollama.generate(model='mistral', prompt=prompt)
    summary = response['response']
    return summary

n_threads = 32
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    summaries = list(executor.map(generate_suggestions, [row for _, row in data.iterrows()]))

data['summary'] = summaries

In [7]:
data['summary']

0       Resumen: Vehículo ID 1, modelo Lotus Esprit d...
1       Resumen: Vehículo ID 2, marca Rolls-Royce, mo...
2       Resumen: Vehículo ID 3, marca Acura, modelo V...
3       Resumen: ID 4, UUID 7dd31212-2267-4fa5-8e15-0...
4       Resumen: Vehículo ID 5, marca Pontiac, modelo...
                             ...                        
495     Resumen: Vehículo ID 496, tipo minivan Merced...
496     Resumen: Vehículo ID 497, marca Ford, modelo ...
497     Resumen: ID: 498, uuid: 29620851-0325-401c-b3...
498     Resumen: Vehículo ID 499, marca Saab, modelo ...
499     Resumen: Vehículo ID 500, UUID 5d56d515-adb8-...
Name: summary, Length: 500, dtype: object

In [8]:
with tempfile.NamedTemporaryFile(mode='w', delete=False) as temp_file:
    data.to_csv(temp_file, index=False)
    temp_filename = temp_file.name

loader = CSVLoader(temp_filename)
loaded_data = loader.load()
os.remove(temp_filename)

# Embedding de los documentos
embedded_doc = embeddings.embed_documents([text.page_content for text in loaded_data])
embedded_doc = np.array(embedded_doc)

index_full_docs = faiss.IndexFlatL2(embedded_doc.shape[1])
index_full_docs.add(embedded_doc)

# Embedding de los resúmenes
embedded_query = embeddings.embed_documents(data['summary'].tolist())
embedded_query = np.array(embedded_query)

index_datos = faiss.IndexFlatL2(embedded_query.shape[1])
index_datos.add(embedded_query)

In [9]:
def buscar_por_tags(user_tags, k=5):
    prompt_usuario = f"Resumen: {user_tags}"
    embedding_user = embeddings.embed_query(prompt_usuario)
    embedding_user = np.array([embedding_user])
    distances, indices = index_datos.search(embedding_user, k)
    resultados = data.iloc[indices[0]]
    return resultados

In [10]:
tags_usuario = input("Introduce los tags de búsqueda (ej: rojo, Toyota, 2018, gasolina): ")
resultados = buscar_por_tags(tags_usuario)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


print(resultados[['summary']])

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  summary
379                                                                                                                                                                                                                                                 Resumen: Vehículo con ID 380 y uuid 4cca54